# 🥈 Silver — Limpeza, SCD, MERGE e Governança

## Features implementadas

| # | Feature | Descrição |
|---|---------|----------|
| 1 | **Change Data Feed (CDF)** | Lê apenas mudanças do Bronze |
| 2 | **SCD Type 1** | MERGE — mantém só versão mais recente |
| 3 | **SCD Type 2** | MERGE com `valid_from / valid_to / is_current` |
| 4 | **MERGE incremental** | Silver principal sem overwrite |
| 5 | `OPTIMIZE + Z-ORDER` | Compacta + co-localiza por colunas quentes |
| 6 | **Bloom Filter Index** | Busca por valor exato em colunas de alta cardinalidade |
| 7 | `VACUUM` | Limpeza de arquivos antigos |
| 8 | **Deletion Vectors** | Deletes eficientes sem reescrever arquivos |
| 9 | **Table Constraints** | `NOT NULL` e `CHECK` — qualidade no nível do banco |
| 10 | **AQE + Broadcast** | Adaptive Query Execution + broadcast hints |
| 11 | **Salting** | Resolve data skew em joins com chave concentrada |
| 12 | **GDPR Delete** | Direito ao esquecimento com `DELETE` + `VACUUM` |
| 13 | **Cache** | Tabelas quentes em memória |
| 14 | **Row-level Security** | Filtro por linha via Unity Catalog |
| 15 | **Column Masking** | Mascaramento de `email` por perfil de acesso |

## 0. Configuração

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = 'workspace'
SCHEMA  = 'medallion_demo'

spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print(f'✅ Usando {CATALOG}.{SCHEMA}')

## 1. Change Data Feed (CDF) — Ler apenas mudanças do Bronze

Cada linha vem com `_change_type`: `insert`, `update_preimage`, `update_postimage`, `delete`.
Para a Silver, interessa apenas `insert` + `update_postimage` (estado mais recente).

In [ ]:
try:
    bronze_changes = (
        spark.read
        .format('delta')
        .option('readChangeData', 'true')
        .option('startingVersion', 1)
        .table('bronze_sales')
    )
    bronze_new = bronze_changes.filter(
        F.col('_change_type').isin('insert', 'update_postimage')
    ).drop('_change_type', '_commit_version', '_commit_timestamp')

    print(f'✅ CDF lido: {bronze_new.count():,} mudanças desde a versão 1')
    display(bronze_changes.groupBy('_change_type').count())

except Exception as e:
    print(f'⚠️  CDF indisponível. Lendo tabela completa. Erro: {e}')
    bronze_new = spark.table('bronze_sales')

## 2. SCD Type 1 — Dimensão Cliente (última versão)

Quando um atributo muda, o registro é **sobrescrito**. Sem histórico.

In [ ]:
spark.sql('''
    CREATE TABLE IF NOT EXISTS dim_customer_scd1 (
        customer_id     INT,
        email           STRING,
        country         STRING,
        payment_method  STRING,
        first_seen      TIMESTAMP,
        last_updated    TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES ("delta.enableChangeDataFeed" = "true")
''')

bronze_full = spark.table('bronze_sales')
w_cust = Window.partitionBy('customer_id').orderBy(F.col('_ingest_ts').desc())

customers_latest = (
    bronze_full
    .withColumn('_rn', F.row_number().over(w_cust))
    .filter(F.col('_rn') == 1)
    .withColumn('email', F.coalesce(F.col('email'), F.lit('desconhecido')))
    .select(
        'customer_id', 'email', 'country', 'payment_method',
        F.col('_ingest_ts').alias('first_seen'),
        F.current_timestamp().alias('last_updated'),
    )
)

(DeltaTable.forName(spark, 'dim_customer_scd1').alias('tgt')
    .merge(customers_latest.alias('src'), 'tgt.customer_id = src.customer_id')
    .whenMatchedUpdate(set={
        'email':          'src.email',
        'country':        'src.country',
        'payment_method': 'src.payment_method',
        'last_updated':   'src.last_updated',
    })
    .whenNotMatchedInsertAll()
    .execute())

print(f'✅ SCD1: {spark.table("dim_customer_scd1").count():,} clientes')

## 3. SCD Type 2 — Dimensão Cliente com histórico completo

Cada mudança gera um novo registro. O anterior é expirado (`is_current = false`).

**Algoritmo (2 MERGEs):**
1. Fecha registros que mudaram (`valid_to = hoje`, `is_current = false`)
2. Insere novos registros correntes

In [ ]:
spark.sql('''
    CREATE TABLE IF NOT EXISTS dim_customer_scd2 (
        surrogate_key   STRING,
        customer_id     INT,
        email           STRING,
        country         STRING,
        payment_method  STRING,
        valid_from      DATE,
        valid_to        DATE,
        is_current      BOOLEAN
    )
    USING DELTA
    TBLPROPERTIES ("delta.enableChangeDataFeed" = "true")
''')
print('✅ dim_customer_scd2 criada')

In [ ]:
incoming = (
    bronze_full
    .withColumn('_rn', F.row_number().over(Window.partitionBy('customer_id').orderBy(F.col('_ingest_ts').desc())))
    .filter(F.col('_rn') == 1)
    .withColumn('email', F.coalesce(F.col('email'), F.lit('desconhecido')))
    .select('customer_id', 'email', 'country', 'payment_method')
)

target_scd2 = DeltaTable.forName(spark, 'dim_customer_scd2')
current_records = target_scd2.toDF().filter('is_current = true')

# Detecta mudanças
changed = (
    incoming.alias('inc')
    .join(current_records.alias('cur'), 'customer_id', 'inner')
    .filter(
        (F.col('inc.email')          != F.col('cur.email'))   |
        (F.col('inc.country')        != F.col('cur.country')) |
        (F.col('inc.payment_method') != F.col('cur.payment_method'))
    )
    .select('inc.customer_id')
)

# PASSO 1: Fecha registros antigos
(target_scd2.alias('tgt')
    .merge(changed.alias('chg'), 'tgt.customer_id = chg.customer_id AND tgt.is_current = true')
    .whenMatchedUpdate(set={'is_current': 'false', 'valid_to': F.current_date()})
    .execute())

# PASSO 2: Insere novos registros correntes
new_current = (
    incoming
    .withColumn('surrogate_key', F.md5(F.concat(F.col('customer_id').cast('string'), F.current_date().cast('string'))))
    .withColumn('valid_from',    F.current_date())
    .withColumn('valid_to',      F.lit('9999-12-31').cast('date'))
    .withColumn('is_current',    F.lit(True))
)

(target_scd2.alias('tgt')
    .merge(new_current.alias('src'), 'tgt.customer_id = src.customer_id AND tgt.is_current = true')
    .whenNotMatchedInsertAll()
    .execute())

total   = spark.table('dim_customer_scd2').count()
current = spark.table('dim_customer_scd2').filter('is_current = true').count()
print(f'✅ SCD2: {total:,} registros ({current:,} correntes, {total-current:,} históricos)')

## 4. Silver principal — DDL + MERGE incremental

In [ ]:
spark.sql('''
    CREATE TABLE IF NOT EXISTS silver_sales (
        transaction_id  STRING,
        customer_id     INT,
        product_id      INT,
        category        STRING,
        country         STRING,
        payment_method  STRING,
        unit_price      DOUBLE,
        quantity        INT,
        total_amount    DOUBLE,
        email           STRING,
        event_ts        TIMESTAMP,
        event_date      DATE
    )
    USING DELTA
    PARTITIONED BY (event_date)
    TBLPROPERTIES (
        "delta.enableChangeDataFeed"         = "true",
        "delta.autoOptimize.optimizeWrite"   = "true",
        "delta.autoOptimize.autoCompact"     = "true",
        "delta.deletedFileRetentionDuration" = "interval 7 days"
    )
''')
print('✅ silver_sales criada')

In [ ]:
w_dedup = Window.partitionBy('transaction_id').orderBy(F.col('_ingest_ts').desc())

silver_df = (
    bronze_full
    .withColumn('_rn', F.row_number().over(w_dedup))
    .filter(F.col('_rn') == 1).drop('_rn')
    .withColumn('event_ts',    F.to_timestamp('event_ts'))
    .withColumn('event_date',  F.to_date('event_ts'))
    .withColumn('category',    F.initcap(F.trim(F.col('category'))))
    .filter((F.col('unit_price') > 0) & (F.col('quantity') > 0))
    .filter(F.col('event_ts').isNotNull())
    .withColumn('total_amount', F.round(F.col('unit_price') * F.col('quantity'), 2))
    .withColumn('email', F.coalesce(F.col('email'), F.lit('desconhecido')))
    .select(
        'transaction_id', 'customer_id', 'product_id', 'category',
        'country', 'payment_method', 'unit_price', 'quantity',
        'total_amount', 'email', 'event_ts', 'event_date'
    )
)

(DeltaTable.forName(spark, 'silver_sales').alias('tgt')
    .merge(silver_df.alias('src'), 'tgt.transaction_id = src.transaction_id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

silver_count = spark.table('silver_sales').count()
print(f'✅ Silver MERGE: {silver_count:,} linhas')

## 5. OPTIMIZE + Z-ORDER BY

- **OPTIMIZE**: compacta arquivos pequenos → menos I/O por leitura
- **Z-ORDER**: co-localiza valores relacionados das colunas especificadas → data skipping eficiente

In [ ]:
spark.sql('OPTIMIZE silver_sales ZORDER BY (country, category, event_date)')
print('✅ OPTIMIZE + Z-ORDER concluído')
display(spark.sql('DESCRIBE HISTORY silver_sales LIMIT 3'))

## 6. Bloom Filter Index

Estrutura probabilística que responde "este arquivo pode conter o valor X?" em O(1).
Ideal para colunas de alta cardinalidade em buscas por igualdade (`WHERE email = 'x'`).

- `fpp` = false positive probability
- `numItems` = cardinalidade estimada

In [ ]:
spark.sql('''
    CREATE BLOOMFILTER INDEX ON TABLE silver_sales
    FOR COLUMNS(
        email          OPTIONS (fpp=0.1,  numItems=200000),
        customer_id    OPTIONS (fpp=0.05, numItems=200000),
        transaction_id OPTIONS (fpp=0.01, numItems=5100000)
    )
''')
print('✅ Bloom Filter criado para email, customer_id, transaction_id')

## 7. VACUUM — Limpeza de arquivos antigos

Remove arquivos Delta não referenciados por nenhuma versão dentro da janela de retenção.

In [ ]:
print('=== DRY RUN ===')
display(spark.sql('VACUUM silver_sales RETAIN 168 HOURS DRY RUN'))

spark.sql('VACUUM silver_sales RETAIN 168 HOURS')
print('✅ VACUUM silver_sales concluído')

for tbl in ['dim_customer_scd1', 'dim_customer_scd2']:
    spark.sql(f'VACUUM {tbl} RETAIN 168 HOURS')
    print(f'✅ VACUUM {tbl} concluído')

## 8. Deletion Vectors — Deletes eficientes

Sem Deletion Vectors: `DELETE` reescreve os arquivos Parquet afetados (caro).

Com Deletion Vectors (DBR 12.1+): o delete é registrado num **vetor de bits** por arquivo.
O arquivo original **não é reescrito** — muito mais rápido. O OPTIMIZE limpa os vetores
fisicamente quando conveniente.

Redução típica: de minutos para segundos em tabelas grandes.

In [ ]:
# Habilita Deletion Vectors na tabela silver
spark.sql('''
    ALTER TABLE silver_sales
    SET TBLPROPERTIES ("delta.enableDeletionVectors" = "true")
''')
print('✅ Deletion Vectors habilitados em silver_sales')

# DELETE agora é registrado como vetor — o arquivo Parquet NÃO é reescrito
before = spark.table('silver_sales').count()
spark.sql("DELETE FROM silver_sales WHERE unit_price < 0.01")
after  = spark.table('silver_sales').count()
print(f'   Deletadas {before - after:,} linhas com preço inválido (via Deletion Vector)')
print(f'   Antes: {before:,} → Depois: {after:,}')

# Após OPTIMIZE os vetores são compactados fisicamente
spark.sql('OPTIMIZE silver_sales')
print('✅ OPTIMIZE limpou os Deletion Vectors fisicamente')

## 9. Table Constraints — Qualidade no nível do banco

O Delta Lake impõe restrições de dados **na escrita** — qualquer INSERT/UPDATE que
viole a constraint é rejeitado com erro, mesmo vindo de MERGE ou Streaming.

| Tipo | Sintaxe | Descrição |
|------|---------|----------|
| `NOT NULL` | `ALTER TABLE ... CHANGE COLUMN ... SET NOT NULL` | Coluna não aceita NULL |
| `CHECK` | `ALTER TABLE ... ADD CONSTRAINT nome CHECK (expr)` | Expressão booleana arbitrária |

In [ ]:
# CHECK: preço deve ser positivo
spark.sql('''
    ALTER TABLE silver_sales
    ADD CONSTRAINT valid_price CHECK (unit_price > 0)
''')

# CHECK: quantidade deve ser >= 1
spark.sql('''
    ALTER TABLE silver_sales
    ADD CONSTRAINT valid_quantity CHECK (quantity >= 1)
''')

# CHECK: total_amount deve ser positivo
spark.sql('''
    ALTER TABLE silver_sales
    ADD CONSTRAINT valid_total CHECK (total_amount > 0)
''')

print('✅ Constraints adicionadas à silver_sales')

# Visualiza as constraints ativas
display(spark.sql("SHOW TBLPROPERTIES silver_sales").filter(F.col('key').contains('constraint')))

# Teste: tenta inserir registro inválido (deve falhar)
try:
    spark.sql('''
        INSERT INTO silver_sales VALUES
        ('TX-BAD', 1, 1, 'Test', 'BR', 'pix', -10.0, 1, -10.0, 'test@mail.com',
         current_timestamp(), current_date())
    ''')
    print('⚠️  Inserção com preço negativo passou — constraint não ativa')
except Exception as e:
    print(f'✅ Constraint bloqueou corretamente o INSERT com preço negativo')
    print(f'   Erro: {str(e)[:120]}')

## 10. AQE (Adaptive Query Execution) + Broadcast Hints

**AQE**: o Spark ajusta o plano de execução em runtime baseado em estatísticas reais.
- `coalescePartitions` → reduz partições vazias/pequenas após shuffle
- `skewJoin` → detecta e mitiga data skew automaticamente

**Broadcast hints**: força o Spark a usar broadcast join quando sabe que uma tabela é pequena.

In [ ]:
# Habilita AQE globalmente na sessão
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.adaptive.advisoryPartitionSizeInBytes', '128m')
print('✅ AQE configurado')

# Configura threshold para broadcast automático (tabelas < 100MB são broadcastadas)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '104857600')  # 100MB

# Broadcast hint explícito: força broadcast da dimensão SCD1 (pequena) no join com silver
silver    = spark.table('silver_sales')
dim_cust  = spark.table('dim_customer_scd1')

# Sem hint: Spark decide o tipo de join (pode ser sort-merge)
# Com hint: força broadcast join (mais rápido quando dim_cust < autoBroadcastJoinThreshold)
enriched = (
    silver
    .join(F.broadcast(dim_cust).alias('dim'), 'customer_id', 'left')  # ← broadcast hint
    .select(
        silver.customer_id,
        silver.transaction_id,
        silver.total_amount,
        F.col('dim.country').alias('customer_country'),
    )
)

print(f'✅ Join com broadcast hint: {enriched.count():,} linhas')
print('   Plano de execução (verifique BroadcastHashJoin):')
enriched.explain(mode='simple')

## 11. Salting — Resolver Data Skew

Data skew acontece quando uma chave de join tem valores muito concentrados
(ex: `customer_id = 1` tem 10x mais transações que a média).
Resultado: um executor fica sobrecarregado enquanto os outros ficam ociosos.

**Salting**: adiciona um sufixo aleatório à chave, distribuindo a carga uniformemente.

```
customer_id=1 (500k linhas) → customer_id_salt='1_0', '1_1', ..., '1_9' (50k cada)
```

In [ ]:
SALT_BUCKETS = 10

# Lado esquerdo (grande): adiciona coluna de sal
silver_salted = (
    silver
    .withColumn('salt', (F.rand() * SALT_BUCKETS).cast('int'))
    .withColumn('customer_id_salt', F.concat(F.col('customer_id').cast('string'), F.lit('_'), F.col('salt').cast('string')))
)

# Lado direito (pequena): expande para todos os valores de sal
dim_exploded = (
    dim_cust
    .crossJoin(spark.range(SALT_BUCKETS).toDF('salt'))
    .withColumn('customer_id_salt', F.concat(F.col('customer_id').cast('string'), F.lit('_'), F.col('salt').cast('string')))
)

# Join pela chave salteada (distribui os executores uniformemente)
result_salted = (
    silver_salted
    .join(dim_exploded, 'customer_id_salt', 'left')
    .drop('salt', 'customer_id_salt')
)

print(f'✅ Salting com {SALT_BUCKETS} buckets aplicado')
print(f'   Resultado: {result_salted.count():,} linhas')
print(f'   customer_id mais frequente (sem salting): {silver.groupBy("customer_id").count().orderBy(F.col("count").desc()).first()[1]:,} transações')

## 12. GDPR Delete — Direito ao esquecimento

O GDPR exige que dados de um usuário sejam apagados **permanentemente** mediante solicitação.

**Fluxo:**
1. `DELETE FROM` em todas as tabelas que contêm o dado
2. `VACUUM RETAIN 0 HOURS` remove os arquivos fisicamente
   (requer desabilitar o safety check de retenção)

> **Atenção**: após o VACUUM com 0h não é possível fazer Time Travel para versões que
> contêm o dado deletado. Isso é intencional — é o objetivo do GDPR.

In [ ]:
GDPR_CUSTOMER_ID = 42  # exemplo: cliente que solicitou exclusão

# Conta registros antes
before_silver = spark.sql(f'SELECT COUNT(*) FROM silver_sales WHERE customer_id = {GDPR_CUSTOMER_ID}').collect()[0][0]
before_scd1   = spark.sql(f'SELECT COUNT(*) FROM dim_customer_scd1 WHERE customer_id = {GDPR_CUSTOMER_ID}').collect()[0][0]
before_scd2   = spark.sql(f'SELECT COUNT(*) FROM dim_customer_scd2 WHERE customer_id = {GDPR_CUSTOMER_ID}').collect()[0][0]
print(f'Registros do cliente {GDPR_CUSTOMER_ID} antes do delete:')
print(f'  silver_sales     : {before_silver}')
print(f'  dim_customer_scd1: {before_scd1}')
print(f'  dim_customer_scd2: {before_scd2}')

# Step 1: DELETE lógico em todas as tabelas
for tbl in ['silver_sales', 'dim_customer_scd1', 'dim_customer_scd2']:
    spark.sql(f'DELETE FROM {tbl} WHERE customer_id = {GDPR_CUSTOMER_ID}')
    print(f'✅ DELETE em {tbl}')

# Step 2: VACUUM para remoção física
# Desabilita o safety check de 7 dias (apenas para GDPR / dados sensíveis)
spark.conf.set('spark.databricks.delta.retentionDurationCheck.enabled', 'false')

for tbl in ['silver_sales', 'dim_customer_scd1', 'dim_customer_scd2']:
    spark.sql(f'VACUUM {tbl} RETAIN 0 HOURS')
    print(f'✅ VACUUM físico em {tbl}')

# Reabilita o safety check
spark.conf.set('spark.databricks.delta.retentionDurationCheck.enabled', 'true')

# Verifica remoção
after_silver = spark.sql(f'SELECT COUNT(*) FROM silver_sales WHERE customer_id = {GDPR_CUSTOMER_ID}').collect()[0][0]
print(f'\n✅ GDPR Delete concluído. Registros remanescentes: {after_silver}')

## 13. Cache de tabelas quentes

Tabelas acessadas com frequência podem ser mantidas em memória/SSD dos executores.
Ideal para dimensões pequenas usadas em muitos joins.

> No Databricks, o cache é gerenciado pelo **Databricks IO Cache** (automático) +
> `spark.catalog.cacheTable()` (explícito).

In [ ]:
# Cache explícito de tabelas frequentemente consultadas
tables_to_cache = ['dim_customer_scd1', 'dim_customer_scd2']

for tbl in tables_to_cache:
    spark.catalog.cacheTable(tbl)
    is_cached = spark.catalog.isCached(tbl)
    print(f'✅ {tbl} cacheada: {is_cached}')

# Verifica via SQL (a query seguinte usa cache)
import time

start = time.time()
cnt = spark.table('dim_customer_scd1').count()
print(f'\n   dim_customer_scd1.count() com cache: {time.time()-start:.3f}s ({cnt:,} linhas)')

# Limpa cache quando não precisar mais (libera memória)
# spark.catalog.uncacheTable('dim_customer_scd1')
# spark.catalog.clearCache()  # limpa tudo
print('\nℹ️  uncacheTable e clearCache comentados — descomente quando necessário')

## 14. Row-level Security (Unity Catalog)

**Row Filter**: cada usuário/grupo vê apenas as linhas permitidas.
O filtro é aplicado automaticamente pelo Unity Catalog — transparente para quem consulta.

**Exemplo**: analistas do Brasil (`group:br-analysts`) veem apenas `country = 'BR'`.

> Requer Unity Catalog habilitado e privilégios de administrador.

In [ ]:
# Row-level Security via Unity Catalog (requer admin)
# Os comandos abaixo são SQL — execute em um SQL cell ou via spark.sql()

rls_sql = '''
-- 1. Cria a função de filtro por país
CREATE OR REPLACE FUNCTION medallion_demo.filter_by_country(country STRING)
RETURNS BOOLEAN
RETURN
    CASE
        -- admin e data engineers veem tudo
        WHEN is_account_group_member('data-engineers') THEN TRUE
        -- analistas BR veem apenas Brasil
        WHEN is_account_group_member('br-analysts') AND country = 'BR' THEN TRUE
        -- analistas US veem apenas Estados Unidos
        WHEN is_account_group_member('us-analysts') AND country = 'US' THEN TRUE
        ELSE FALSE
    END;

-- 2. Aplica o Row Filter na tabela
ALTER TABLE silver_sales
SET ROW FILTER medallion_demo.filter_by_country ON (country);
'''

print('ℹ️  Row-level Security SQL (execute com conta de admin):')
print(rls_sql)

# Para remover o filtro:
# spark.sql('ALTER TABLE silver_sales DROP ROW FILTER')

# Testa se o filtro está ativo
try:
    spark.sql(rls_sql)
    print('✅ Row Filter aplicado em silver_sales')
except Exception as e:
    print(f'⚠️  Row Filter requer admin: {str(e)[:80]}')

## 15. Column Masking (Unity Catalog)

**Column Mask**: usuários sem permissão veem a coluna mascarada (ex: `u***@mail.com`).
Usuários privilegiados veem o valor real.

Aplicado automaticamente pelo Unity Catalog — a tabela tem uma única definição,
o mascaramento varia por usuário/grupo.

In [ ]:
masking_sql = '''
-- 1. Cria função de mascaramento de email
CREATE OR REPLACE FUNCTION medallion_demo.mask_email(email STRING)
RETURNS STRING
RETURN
    CASE
        -- data-engineers veem o email real
        WHEN is_account_group_member('data-engineers') THEN email
        -- demais: mascara tudo exceto 1o caractere e domínio
        ELSE regexp_replace(email, '(^.)([^@]+)', '$1***')
    END;

-- 2. Aplica a máscara na coluna email da silver_sales
ALTER TABLE silver_sales
ALTER COLUMN email
SET MASK medallion_demo.mask_email;
'''

print('ℹ️  Column Masking SQL (execute com conta de admin):')
print(masking_sql)

# Para remover a máscara:
# spark.sql('ALTER TABLE silver_sales ALTER COLUMN email DROP MASK')

try:
    spark.sql(masking_sql)
    print('✅ Column Mask aplicado em silver_sales.email')
except Exception as e:
    print(f'⚠️  Column Masking requer admin: {str(e)[:80]}')

# Demo do efeito: simula o mascaramento localmente
print('\n=== Efeito do mascaramento (simulado localmente) ===')
demo = spark.createDataFrame(
    [('user123@mail.com',), ('admin@corp.com',), ('cliente456@gmail.com',)],
    ['email']
)
display(
    demo.withColumn('email_masked', F.regexp_replace(F.col('email'), '(^.)([^@]+)', '$1***'))
)

## 16. Relatório final de qualidade

In [ ]:
silver = spark.table('silver_sales')
total  = silver.count()

metrics = {
    'total_linhas':          total,
    'clientes_unicos':       silver.select('customer_id').distinct().count(),
    'transacoes_unicas':     silver.select('transaction_id').distinct().count(),
    'emails_conhecidos_pct': round(silver.filter(F.col('email') != 'desconhecido').count() / total * 100, 2),
    'data_min':              silver.agg(F.min('event_date').cast('string')).collect()[0][0],
    'data_max':              silver.agg(F.max('event_date').cast('string')).collect()[0][0],
    'receita_total':         round(silver.agg(F.sum('total_amount')).collect()[0][0], 2),
    'ticket_medio':          round(silver.agg(F.avg('total_amount')).collect()[0][0], 2),
}

print('=== Qualidade Silver Sales ===')
for k, v in metrics.items():
    print(f'  {k:<30s}: {v:,}')

display(silver.groupBy('country').agg(
    F.count('*').alias('transacoes'),
    F.round(F.sum('total_amount'), 2).alias('receita'),
).orderBy(F.col('receita').desc()))